## Semana 2 Dia 3

Agora vamos a mais detalhes:

1. Diferentes modelos

2. Saídas estruturadas

3. Guardrails

In [1]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, input_guardrail, GuardrailFunctionOutput
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from pydantic import BaseModel

In [2]:
load_dotenv(override=True)

True

In [3]:
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"A chave da API da OpenAI existe e começa com {openai_api_key[:8]}")
else:
    print("Chave de API da OpenAI não definida")

if google_api_key:
    print(f"A chave da API do Google existe e começa com {google_api_key[:2]}")
else:
    print("Chave de API do Google não definida (e isto é opcional)")

if deepseek_api_key:
    print(f"A chave da API da DeepSeek existe e começa com {deepseek_api_key[:3]}")
else:
    print("Chave de API da DeepSeek não definida (e isto é opcional)")

if groq_api_key:
    print(f"A chave da API da Groq existe e começa com {groq_api_key[:4]}")
else:
    print("Chave de API da Groq não definida (e isto é opcional)")

A chave da API da OpenAI existe e começa com sk-proj-
A chave da API do Google existe e começa com AI
A chave da API da DeepSeek existe e começa com sk-
A chave da API da Groq existe e começa com gsk_


In [ ]:
instructions1 = "Você é um agente de vendas que trabalha para a ComplAI, \
uma empresa que fornece uma ferramenta SaaS para garantir a conformidade com SOC2 e preparar auditorias, impulsionada por IA. \
Você escreve cold emails profissionais e sérios."

instructions2 = "Você é um agente de vendas bem-humorado e envolvente que trabalha para a ComplAI, \
uma empresa que fornece uma ferramenta SaaS para garantir a conformidade com SOC2 e preparar auditorias, impulsionada por IA. \
Você escreve cold emails espirituosos e envolventes, com grande chance de obter resposta."

instructions3 = "Você é um agente de vendas ocupado que trabalha para a ComplAI, \
uma empresa que fornece uma ferramenta SaaS para garantir a conformidade com SOC2 e preparar auditorias, impulsionada por IA. \
Você escreve cold emails concisos e diretos ao ponto."

### É fácil usar quaisquer modelos com endpoints compatíveis com OpenAI

In [5]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

In [6]:

deepseek_client = AsyncOpenAI(base_url=DEEPSEEK_BASE_URL, api_key=deepseek_api_key)
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)

deepseek_model = OpenAIChatCompletionsModel(model="deepseek-chat", openai_client=deepseek_client)
gemini_model = OpenAIChatCompletionsModel(model="gemini-2.0-flash", openai_client=gemini_client)
llama3_3_model = OpenAIChatCompletionsModel(model="llama-3.3-70b-versatile", openai_client=groq_client)

In [7]:
sales_agent1 = Agent(name="Agente de Vendas DeepSeek", instructions=instructions1, model=deepseek_model)
sales_agent2 =  Agent(name="Agente de Vendas Gemini", instructions=instructions2, model=gemini_model)
sales_agent3  = Agent(name="Agente de Vendas Llama3.3",instructions=instructions3,model=llama3_3_model)

In [8]:
description = "Escreva um cold email de vendas"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

In [9]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Enviar um e-mail com o assunto e o corpo HTML informados para todos os prospects de vendas """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("contato@santsoft.com")  # Altere para o seu remetente verificado
    to_email = To("santclear@gmail.com")  # Altere para o seu destinatário
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [10]:
subject_instructions = "Você pode escrever um assunto para um cold email de vendas. \
Você receberá uma mensagem e precisará escrever um assunto para um e-mail que tenha grande chance de obter resposta."

html_instructions = "Você pode converter um corpo de e-mail em texto para um corpo de e-mail em HTML. \
Você receberá um corpo de e-mail em texto que pode conter algum markdown \
e precisará convertê-lo em um corpo de e-mail em HTML com layout e design simples, claros e atraentes."

subject_writer = Agent(name="Redator de assunto de e-mail", instructions=subject_instructions, model="gpt-5-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Escreva um assunto para um cold email de vendas")

html_converter = Agent(name="Conversor de corpo de e-mail HTML", instructions=html_instructions, model="gpt-5-mini")
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Converter um corpo de e-mail em texto para um corpo de e-mail em HTML")

In [11]:
email_tools = [subject_tool, html_tool, send_html_email]

In [12]:
instructions ="Você é um formatador e remetente de e-mails. Você recebe o corpo de um e-mail a ser enviado. \
Primeiro, use a ferramenta subject_writer para escrever um assunto para o e-mail; depois, use a ferramenta html_converter para converter o corpo para HTML. \
Por fim, use a ferramenta send_html_email para enviar o e-mail com o assunto e o corpo em HTML."


emailer_agent = Agent(
    name="Gerente de E-mail",
    instructions=instructions,
    tools=email_tools,
    model="gpt-5-mini",
    handoff_description="Converter um e-mail para HTML e enviá-lo")

In [13]:
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]

In [14]:
sales_manager_instructions = """
Você é um Gerente de Vendas na ComplAI. Seu objetivo é encontrar o único melhor cold email de vendas usando as ferramentas sales_agent.
 
Siga estes passos com atenção:
1. Gerar rascunhos: Use todas as três ferramentas sales_agent para gerar três rascunhos de e-mail diferentes. Não prossiga até que os três rascunhos estejam prontos.
 
2. Avaliar e selecionar: Revise os rascunhos e escolha o único melhor e-mail, usando seu julgamento sobre qual é o mais eficaz.
Você pode usar as ferramentas várias vezes se não ficar satisfeito(a) com os resultados da primeira tentativa.
 
3. Encaminhar para envio: Passe SOMENTE o rascunho vencedor para o agente 'Gerente de E-mail'. O Gerente de E-mail cuidará da formatação e do envio.
 
Regras cruciais:
- Você deve usar as ferramentas de agente de vendas para gerar os rascunhos — não os escreva você mesmo(a).
- Você deve encaminhar exatamente UM e-mail ao Gerente de E-mail — nunca mais de um.
"""


sales_manager = Agent(
    name="Gerente de Vendas",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model="gpt-5-mini")

message = "Envie um cold email de vendas iniciado com 'Prezado CEO' e assinado por Alice"

with trace("SDR Automatizado"):
    result = await Runner.run(sales_manager, message)

## Confira o trace:

https://platform.openai.com/traces

In [15]:
class NameCheckOutput(BaseModel):
    is_name_in_message: bool
    name: str

guardrail_agent = Agent( 
    name="Verificação de nome",
    instructions="Verifique se o usuário está incluindo o nome pessoal de alguém no que ele quer que você faça.",
    output_type=NameCheckOutput,
    model="gpt-5-mini"
)

In [16]:
@input_guardrail
async def guardrail_against_name(ctx, agent, message):
    result = await Runner.run(guardrail_agent, message, context=ctx.context)
    is_name_in_message = result.final_output.is_name_in_message
    return GuardrailFunctionOutput(output_info={"found_name": result.final_output},tripwire_triggered=is_name_in_message)

In [17]:
careful_sales_manager = Agent(
    name="Gerente de Vendas",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=[emailer_agent],
    model="gpt-5-mini",
    input_guardrails=[guardrail_against_name]
    )

message = "Envie um cold email de vendas iniciado com 'Prezado CEO' e assinado por Alice"

with trace("SDR Automatizado Protegido"):
    result = await Runner.run(careful_sales_manager, message)

InputGuardrailTripwireTriggered: Guardrail InputGuardrail triggered tripwire

## Confira o trace:

https://platform.openai.com/traces

In [18]:

message = "Envie um cold email de vendas iniciado com 'Prezado CEO' e assinado por Diretor de Desenvolvimento de Negócios"

with trace("SDR Automatizado Protegido"):
    result = await Runner.run(careful_sales_manager, message)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercício</h2>
            <span style="color:#ff7800;">• Experimente modelos diferentes<br/>• Adicione mais guardrails de entrada e saída<br/>• Use saídas estruturadas para a geração do e-mail
            </span>
        </td>
    </tr>
</table>